# Service Account Inventory & Lifecycle Audit Notebook

Portable service-account inventory for a new Snowflake environment. Run top to bottom. It
surfaces:

1. **Full service account inventory** — every likely service account, its auth method, owner, and status
2. **Authentication method breakdown** — password-only vs. key-pair vs. MFA-capable, account-wide
3. **Role assignments** — what each service account can actually do, and whether any hold admin-level roles
4. **Activity & staleness** — login history, dormant accounts, and MFA usage in practice (not just configured)
5. **Key rotation signal** — accounts with a single long-lived key vs. two active keys (mid-rotation)
6. **Targeted network policy check** — a Python cell that checks account/user-level network policy only for the accounts flagged highest-risk by earlier sections, rather than looping every user in the account
7. **A rolled-up per-account lifecycle/risk summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database (or `ACCOUNTADMIN`).
- Section 6 runs live `SHOW PARAMETERS` / `SHOW NETWORK POLICIES` calls via Python, so it needs
  a role that can actually see those objects (typically `SECURITYADMIN` or `ACCOUNTADMIN`).
- Nothing here is destructive — every cell reads. Nothing writes to the account.

### A note on identifying "service accounts"
Only recent Snowflake accounts reliably populate `USERS.TYPE` with `SERVICE` /
`LEGACY_SERVICE`. Where that's blank, this notebook falls back to naming-convention matching
(`%SVC%`, `%SERVICE%`, `%_BOT%`, `%_APP%`, `%_ETL%`, `%_INTEGRATION%`) — treat that fallback
list as a starting point to confirm with the team, not a definitive inventory. It's worth
asking early whether the org has a naming standard for service accounts; if so, swap it into
the `service_account_name_patterns` parameter below.


In [ ]:
-- ============================================================
-- PARAMETERS — adjust once per environment.
-- ============================================================
SET admin_roles = 'ACCOUNTADMIN,SECURITYADMIN,SYSADMIN,ORGADMIN';
SET stale_login_days = 90;          -- flag accounts with no successful login in this many days
SET service_account_name_patterns = '%SVC%,%SERVICE%,%_BOT%,%_APP%,%_ETL%,%_INTEGRATION%,%_PIPELINE%';

SELECT $admin_roles AS admin_roles,
       $stale_login_days AS stale_login_days,
       $service_account_name_patterns AS service_account_name_patterns;


## 1. Full service account inventory

Every user that is either explicitly typed as a service account, or matches a service-account
naming pattern. This is the base table the rest of the notebook filters and joins against.


In [ ]:
-- Base inventory of likely service accounts
SELECT
    name AS user_name,
    type AS user_type,
    disabled,
    disabled_reason,
    owner,
    default_role,
    default_warehouse,
    has_password,
    has_rsa_public_key,
    must_change_password,
    ext_authn_duo,
    last_success_login,
    DATEDIFF('day', last_success_login, CURRENT_TIMESTAMP()) AS days_since_last_login,
    days_to_expiry,
    created_on,
    comment
FROM snowflake.account_usage.users
WHERE deleted_on IS NULL
  AND (
        type IN ('SERVICE', 'LEGACY_SERVICE')
        OR EXISTS (
            SELECT 1
            FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
            WHERE users.name ILIKE p.value
        )
      )
ORDER BY user_type NULLS LAST, user_name;


## 2. Authentication method breakdown

Password-only service accounts (no key-pair, no MFA) are the classic finding here — they're
the accounts to prioritize for migration to key-pair or OAuth first, since a leaked static
password is the easiest credential to compromise and the hardest to detect misuse of.


In [ ]:
-- Auth method distribution across all likely service accounts
WITH svc AS (
    SELECT *
    FROM snowflake.account_usage.users
    WHERE deleted_on IS NULL
      AND (
            type IN ('SERVICE', 'LEGACY_SERVICE')
            OR EXISTS (
                SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
                WHERE users.name ILIKE p.value
            )
          )
)
SELECT
    CASE
        WHEN has_password AND NOT has_rsa_public_key THEN 'PASSWORD ONLY'
        WHEN has_rsa_public_key AND NOT has_password THEN 'KEY-PAIR ONLY'
        WHEN has_password AND has_rsa_public_key THEN 'PASSWORD + KEY-PAIR (both active)'
        ELSE 'NEITHER (likely OAuth/external or unconfigured)'
    END AS auth_method,
    COUNT(*) AS account_count,
    SUM(IFF(disabled, 1, 0)) AS disabled_count
FROM svc
GROUP BY auth_method
ORDER BY account_count DESC;


In [ ]:
-- The password-only list itself, for direct follow-up
SELECT
    name AS user_name,
    type AS user_type,
    disabled,
    owner,
    default_role,
    last_success_login,
    created_on
FROM snowflake.account_usage.users
WHERE deleted_on IS NULL
  AND has_password = TRUE
  AND has_rsa_public_key = FALSE
  AND (
        type IN ('SERVICE', 'LEGACY_SERVICE')
        OR EXISTS (
            SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
            WHERE users.name ILIKE p.value
        )
      )
ORDER BY user_name;


## 3. Role assignments — what each service account can actually do

The account itself matters less than what it's authorized to do. A service account holding
an admin-level role is a standing finding worth a direct conversation with its owner about
whether that scope is actually needed.


In [ ]:
-- Roles held by each service account
WITH svc AS (
    SELECT name FROM snowflake.account_usage.users
    WHERE deleted_on IS NULL
      AND (
            type IN ('SERVICE', 'LEGACY_SERVICE')
            OR EXISTS (
                SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
                WHERE users.name ILIKE p.value
            )
          )
)
SELECT
    gu.grantee_name AS service_account,
    gu.role,
    IFF(gu.role IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($admin_roles, ','))), 'YES', 'no') AS is_admin_role,
    gu.granted_by,
    gu.created_on AS granted_on
FROM snowflake.account_usage.grants_to_users gu
JOIN svc ON gu.grantee_name = svc.name
WHERE gu.deleted_on IS NULL
ORDER BY (is_admin_role = 'YES') DESC, service_account, role;


## 4. Activity & staleness

Two different questions worth separating: *is this account still being used at all* (dormant
accounts are cleanup candidates), and *when it is used, does it use MFA/key-pair the way it's
configured to* (a configured-but-unused control isn't actually protecting anything).


In [ ]:
-- Dormant service accounts — configured but not logging in
SELECT
    name AS user_name,
    type AS user_type,
    disabled,
    last_success_login,
    DATEDIFF('day', last_success_login, CURRENT_TIMESTAMP()) AS days_since_last_login,
    created_on
FROM snowflake.account_usage.users
WHERE deleted_on IS NULL
  AND (
        type IN ('SERVICE', 'LEGACY_SERVICE')
        OR EXISTS (
            SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
            WHERE users.name ILIKE p.value
        )
      )
  AND (last_success_login IS NULL OR last_success_login < DATEADD(day, -$stale_login_days, CURRENT_TIMESTAMP()))
ORDER BY last_success_login NULLS FIRST;


In [ ]:
-- Recent login activity and authentication factors actually used, per service account
SELECT
    user_name,
    first_authentication_factor,
    second_authentication_factor,
    COUNT(*)                                   AS login_events,
    SUM(IFF(is_success, 1, 0))                 AS successful_logins,
    SUM(IFF(NOT is_success, 1, 0))              AS failed_logins,
    MIN(event_timestamp)                         AS first_seen,
    MAX(event_timestamp)                         AS last_seen,
    COUNT(DISTINCT client_ip)                    AS distinct_source_ips
FROM snowflake.account_usage.login_history
WHERE event_timestamp >= DATEADD(day, -90, CURRENT_TIMESTAMP())
  AND user_name IN (
        SELECT name FROM snowflake.account_usage.users
        WHERE deleted_on IS NULL
          AND (
                type IN ('SERVICE', 'LEGACY_SERVICE')
                OR EXISTS (
                    SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
                    WHERE users.name ILIKE p.value
                )
              )
      )
GROUP BY user_name, first_authentication_factor, second_authentication_factor
ORDER BY user_name, login_events DESC;


**Watch for:** a service account authenticating from an unexpectedly large number of
distinct source IPs (`distinct_source_ips`), which is worth confirming is expected (e.g. a
distributed pipeline) rather than assumed benign. Also watch `failed_logins` — a service
account with a high failure rate may indicate a stale credential still configured somewhere,
or an active attempt to use it.


## 5. Key rotation signal

`ACCOUNT_USAGE.USERS` only exposes whether *a* key is configured (`HAS_RSA_PUBLIC_KEY`), not
whether there are two keys active at once (the state you want during a rotation window, so the
old key keeps working until every consumer has cut over to the new one). `SHOW USERS` exposes
both key fingerprints directly, so this section uses that instead.


In [ ]:
-- SHOW USERS exposes both RSA key fingerprints (needed to see mid-rotation state)
SHOW USERS;


In [ ]:
-- Parse the SHOW USERS output for key-pair rotation status on service accounts
SELECT
    "name" AS user_name,
    "type" AS user_type,
    "has_rsa_public_key",
    "disabled"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "type" IN ('SERVICE', 'LEGACY_SERVICE')
   OR EXISTS (
        SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
        WHERE "name" ILIKE p.value
      )
ORDER BY user_name;


If `SHOW USERS` in this environment exposes `rsa_public_key_fp` and `rsa_public_key_2_fp`
columns directly (naming varies slightly by Snowflake release), select those instead to see
whether a second key is currently populated — that's the clearest "actively mid-rotation"
signal, versus a single key that's simply never been rotated since account creation.


## 6. Targeted network policy check (Python)

Network policies restrict which IP ranges an account can authenticate from — a meaningful
control specifically for service accounts, which should usually only ever connect from a
known, narrow set of infrastructure. Looping `SHOW PARAMETERS ... IN USER` over every user in
the account is wasteful; this checks only the accounts Sections 2–4 already flagged as
higher-risk (password-only, admin-role-holding, or dormant), so the check scales with what
actually needs attention rather than with account size.


In [ ]:
# Targeted network policy check for the highest-risk service accounts identified above.
# Requires a role that can see network policies (typically SECURITYADMIN or ACCOUNTADMIN).

from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Pull the flagged accounts from the SQL cells above by re-querying the same logic directly,
# so this cell is safe to re-run independently of cell execution order.
flagged_df = session.sql('''
    SELECT DISTINCT name AS user_name
    FROM snowflake.account_usage.users
    WHERE deleted_on IS NULL
      AND (
            type IN ('SERVICE', 'LEGACY_SERVICE')
            OR EXISTS (
                SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
                WHERE users.name ILIKE p.value
            )
          )
      AND (
            (has_password = TRUE AND has_rsa_public_key = FALSE)                              -- password-only
            OR last_success_login < DATEADD(day, -$stale_login_days, CURRENT_TIMESTAMP())       -- dormant
            OR last_success_login IS NULL
            OR name IN (
                SELECT grantee_name FROM snowflake.account_usage.grants_to_users
                WHERE deleted_on IS NULL
                  AND role IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($admin_roles, ',')))       -- admin role holder
            )
          )
''').collect()

results = []
for row in flagged_df:
    user_name = row["USER_NAME"]
    try:
        params = session.sql(f'SHOW PARAMETERS LIKE \'NETWORK_POLICY\' IN USER "{user_name}"').collect()
        policy_value = params[0]["value"] if params else None
    except Exception as e:
        policy_value = f"ERROR: {e}"
    results.append({"user_name": user_name, "network_policy": policy_value})

import pandas as pd
result_df = pd.DataFrame(results)
result_df


In [ ]:
-- Account-level default network policy and the full list of defined policies, for context
SHOW NETWORK POLICIES;


An empty or account-default `network_policy` value on a flagged service account is worth
a direct follow-up — service accounts are usually the easiest category to scope down to a
specific IP allow-list, since (unlike a person) they don't need to connect from arbitrary
locations.


## 7. Rolled-up per-account lifecycle/risk summary


In [ ]:
-- One row per service account with every risk signal from this notebook combined
WITH svc AS (
    SELECT *
    FROM snowflake.account_usage.users
    WHERE deleted_on IS NULL
      AND (
            type IN ('SERVICE', 'LEGACY_SERVICE')
            OR EXISTS (
                SELECT 1 FROM TABLE(SPLIT_TO_TABLE($service_account_name_patterns, ',')) p
                WHERE users.name ILIKE p.value
            )
          )
),
admin_holders AS (
    SELECT DISTINCT grantee_name AS user_name
    FROM snowflake.account_usage.grants_to_users
    WHERE deleted_on IS NULL
      AND role IN (SELECT value FROM TABLE(SPLIT_TO_TABLE($admin_roles, ',')))
),
role_counts AS (
    SELECT grantee_name AS user_name, COUNT(*) AS role_count
    FROM snowflake.account_usage.grants_to_users
    WHERE deleted_on IS NULL
    GROUP BY grantee_name
)
SELECT
    s.name AS user_name,
    s.type AS user_type,
    s.disabled,
    s.last_success_login,
    DATEDIFF('day', s.last_success_login, CURRENT_TIMESTAMP()) AS days_since_last_login,
    COALESCE(rc.role_count, 0) AS role_count,
    IFF(s.name IN (SELECT user_name FROM admin_holders), 'YES', 'no') AS holds_admin_role,
    IFF(s.has_password AND NOT s.has_rsa_public_key, 'YES', 'no') AS password_only,
    IFF(s.last_success_login IS NULL OR s.last_success_login < DATEADD(day, -$stale_login_days, CURRENT_TIMESTAMP()), 'YES', 'no') AS dormant,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        IFF(s.name IN (SELECT user_name FROM admin_holders), 'REVIEW ADMIN ROLE NEED', NULL),
        IFF(s.has_password AND NOT s.has_rsa_public_key, 'MIGRATE TO KEY-PAIR/OAUTH', NULL),
        IFF(s.last_success_login IS NULL OR s.last_success_login < DATEADD(day, -$stale_login_days, CURRENT_TIMESTAMP()), 'CONFIRM STILL NEEDED / DEPROVISION', NULL),
        IFF(NOT s.disabled AND s.last_success_login IS NULL, 'NEVER LOGGED IN — VERIFY INTENDED USE', NULL)
    ), '; ') AS recommendations
FROM svc s
LEFT JOIN role_counts rc ON s.name = rc.user_name
ORDER BY (holds_admin_role = 'YES') DESC, (password_only = 'YES') DESC, (dormant = 'YES') DESC, s.name;


### Notes, caveats, and next steps

- **Naming-pattern detection needs a sanity check with the team.** Before treating Section 1's
  list as complete, ask Data Engineering/Platform Services whether there's a naming standard
  for service accounts and adjust `service_account_name_patterns` accordingly — a mismatched
  pattern will both miss real service accounts and pull in unrelated human accounts.
- **This inventory doesn't distinguish "still needed" from "still active."** An account can be
  actively logging in and still be a candidate for consolidation if, say, three separate
  pipelines each got their own service account for the same job over time. That's a
  conversation with the owning team, not something this notebook can determine on its own.
- **Pair this with the RBAC audit notebook's Section 2 and 3** (admin role membership, broad
  grants) — a service account showing up in both notebooks' high-risk lists is the clearest
  priority order for the first remediation pass.
- **OAuth/external auth isn't directly visible in `ACCOUNT_USAGE.USERS`.** An account with
  `has_password = FALSE` and `has_rsa_public_key = FALSE` may be using OAuth, SAML, or simply
  be unconfigured/unused — `LOGIN_HISTORY.FIRST_AUTHENTICATION_FACTOR` in Section 4 is the
  most reliable way to confirm which, since it records what was actually used at login time.
